In [0]:
# Setup widgets and input parameters
dbutils.widgets.text("ClientContainer", "", "Schema/Database Name")
dbutils.widgets.text("TargetCatalog", "main", "Target Catalog")
dbutils.widgets.text("TargetSchema", "", "Target Schema (optional)")

# Get widget values
clientCode = dbutils.widgets.get("ClientContainer")
clientContainer = clientCode.lower() if clientCode else "default"
target_catalog = dbutils.widgets.get("TargetCatalog")
target_schema = dbutils.widgets.get("TargetSchema") or clientContainer

print(f"Source Schema: {clientContainer}")
print(f"Target Location: {target_catalog}.{target_schema}")

In [0]:
from pyspark.sql.functions import col, concat_ws, lit, current_timestamp
from datetime import datetime

In [0]:
# List all tables from the specified schema/database
print(f"Listing tables in schema: {clientContainer}")

try:
    # Get list of tables in the schema
    database_tables = spark.catalog.listTables(clientContainer)
    
    # Convert to DataFrame for easier manipulation
    df_all_tables = spark.createDataFrame(database_tables)
    
    print(f"Found {df_all_tables.count()} total tables in {clientContainer}")
    display(df_all_tables)
except Exception as e:
    print(f"Error listing tables: {str(e)}")
    print(f"Make sure schema '{clientContainer}' exists")
    dbutils.notebook.exit(f"ERROR: Could not list tables in {clientContainer}")

In [0]:
# Filter for tables that start with 'gold_' or 'platinum_'
df_filtered = df_all_tables.filter(
    (col("name").like("gold_%")) | (col("name").like("platinum_%"))
)

print(f"Found {df_filtered.count()} gold/platinum tables")
display(df_filtered)

In [0]:
# Process each gold/platinum table and create copies in target location
results = []
tables_to_process = df_filtered.collect()

for table in tables_to_process:
    table_name = table["name"]
    source_db = table["database"] if "database" in table.asDict() else clientContainer
    full_source_name = f"{source_db}.{table_name}"
    full_target_name = f"{target_catalog}.{target_schema}.{table_name}"
    
    try:
        print(f"\nProcessing: {full_source_name}")
        
        # Read the source table
        df_source = spark.table(full_source_name)
        row_count = df_source.count()
        
        # Add processing metadata
        df_with_metadata = df_source.withColumn("_processed_timestamp", current_timestamp())
        
        # Write to target location as Delta table (overwrite mode)
        df_with_metadata.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(full_target_name)
        
        result = {
            "source_table": full_source_name,
            "target_table": full_target_name,
            "status": "SUCCESS",
            "row_count": row_count,
            "error_message": ""
        }
        print(f"✓ Successfully processed {full_source_name} ({row_count} rows)")
        
    except Exception as e:
        result = {
            "source_table": full_source_name,
            "target_table": full_target_name,
            "status": "FAILED",
            "row_count": 0,
            "error_message": str(e)
        }
        print(f"✗ Failed to process {full_source_name}: {str(e)}")
    
    results.append(result)

# Create summary DataFrame
df_results = spark.createDataFrame(results)
print(f"\n{'='*60}")
print(f"Processing Complete")
print(f"{'='*60}")
display(df_results)

In [0]:
# Build return summary
success_count = len([r for r in results if r["status"] == "SUCCESS"])
failed_count = len([r for r in results if r["status"] == "FAILED"])
total_rows = sum([r["row_count"] for r in results])

return_message = f"Processed {len(results)} tables: {success_count} succeeded, {failed_count} failed. Total rows: {total_rows}"

print(f"\n{return_message}")
dbutils.notebook.exit(return_message)